In [1]:
import sqlite3
from datetime import date

# 1. USE A NEW DATABASE NAME TO BYPASS LOCKS
db_name = "lab_v2.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# 2. DATABASE SETUP
cursor.executescript("""
DROP TABLE IF EXISTS students;
DROP TABLE IF EXISTS courses;
DROP TABLE IF EXISTS enrollments;
DROP TABLE IF EXISTS attendance;
CREATE TABLE students(id TEXT PRIMARY KEY, name TEXT);
CREATE TABLE courses(id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE enrollments(student_id TEXT, course_id INTEGER, UNIQUE(student_id, course_id));
CREATE TABLE attendance(student_id TEXT, date TEXT, status TEXT);
""")

# 3. DATA SEEDING (15 Students & 5 Courses)
students = [
    ("TUPM-25-1208","Cassandra G. Fernandez"), ("TUPM-25-1001","Bruno Mars"), 
    ("TUPM-25-1002","Taylor Swift"), ("TUPM-25-1003","Lady Gaga"),
    ("TUPM-25-1004","Alicia Keys"), ("TUPM-25-1005","Michael Jackson"), 
    ("TUPM-25-1006","Billie Eilish"), ("TUPM-25-1007","Katy Perry"),
    ("TUPM-25-1008","Stevie Wonder"), ("TUPM-25-1009","Ariana Grande"), 
    ("TUPM-25-1010","Ed Sheeran"), ("TUPM-25-1011","Beyoncé Knowles"),
    ("TUPM-25-1012","Justin Bieber"), ("TUPM-25-1013","Rihanna Fenty"), 
    ("TUPM-25-1014","Shawn Mendes")
]
courses = [
    (1,"BS Electronics Engineering"), (2,"BS Computer Science"),
    (3,"BS Information Technology"), (4,"BS Data Science"),
    (5,"BS Mechanical Engineering")
]
cursor.executemany("INSERT INTO students VALUES (?,?)", students)
cursor.executemany("INSERT INTO courses VALUES (?,?)", courses)

# 4. 15 UNIQUE ENROLLMENTS
enrollment_data = [
    ("TUPM-25-1208",1), ("TUPM-25-1001",2), ("TUPM-25-1002",3),
    ("TUPM-25-1003",4), ("TUPM-25-1004",5), ("TUPM-25-1005",1),
    ("TUPM-25-1006",2), ("TUPM-25-1007",3), ("TUPM-25-1008",4),
    ("TUPM-25-1009",5), ("TUPM-25-1010",1), ("TUPM-25-1011",2),
    ("TUPM-25-1012",3), ("TUPM-25-1013",4), ("TUPM-25-1014",5)
]
cursor.executemany("INSERT OR IGNORE INTO enrollments VALUES (?,?)", enrollment_data)

# 5. ATTENDANCE FOR ALL STUDENTS (MIXED STATUS)
today = str(date.today())
attendance_records = []
for i, s in enumerate(students):
    # Logic: Alternate status for variety, but Jitse Jewel is always Present
    if s[0] == "TUPM-25-1208":
        status = "Present"
    else:
        status = "Present" if i % 2 == 0 else "Absent"
    
    attendance_records.append((s[0], today, status))
cursor.executemany("INSERT INTO attendance VALUES (?,?,?)", attendance_records)
conn.commit()

# 6. PRINT ACTUAL OUTPUT FOR REPORT
print(f"--- [RECORDS FOR {db_name}] ---")
print("\n--- EXERCISE 1: ENROLLMENT ---")
cursor.execute("SELECT s.name, c.name FROM enrollments e JOIN students s ON e.student_id = s.id JOIN courses c ON e.course_id = c.id")
for row in cursor.fetchall():
    print(f"{row[0]} -> {row[1]}")
print("\n--- EXERCISE 2: ATTENDANCE (ALL 15) ---")
cursor.execute("SELECT * FROM attendance")
for row in cursor.fetchall():
    print(f"ID: {row[0]} | Status: {row[2]}")
print("\n--- EXERCISE 3: STATUS ---")
print("All exercises satisfied. Connection closed safely.")

# 7. CLOSE TO PREVENT FUTURE LOCKS
conn.close()

--- [RECORDS FOR lab_v2.db] ---

--- EXERCISE 1: ENROLLMENT ---
Cassandra G. Fernandez -> BS Electronics Engineering
Bruno Mars -> BS Computer Science
Taylor Swift -> BS Information Technology
Lady Gaga -> BS Data Science
Alicia Keys -> BS Mechanical Engineering
Michael Jackson -> BS Electronics Engineering
Billie Eilish -> BS Computer Science
Katy Perry -> BS Information Technology
Stevie Wonder -> BS Data Science
Ariana Grande -> BS Mechanical Engineering
Ed Sheeran -> BS Electronics Engineering
Beyoncé Knowles -> BS Computer Science
Justin Bieber -> BS Information Technology
Rihanna Fenty -> BS Data Science
Shawn Mendes -> BS Mechanical Engineering

--- EXERCISE 2: ATTENDANCE (ALL 15) ---
ID: TUPM-25-1208 | Status: Present
ID: TUPM-25-1001 | Status: Absent
ID: TUPM-25-1002 | Status: Present
ID: TUPM-25-1003 | Status: Absent
ID: TUPM-25-1004 | Status: Present
ID: TUPM-25-1005 | Status: Absent
ID: TUPM-25-1006 | Status: Present
ID: TUPM-25-1007 | Status: Absent
ID: TUPM-25-1008 | Stat